In [ ]:
dbutils.widgets.text("env", "dev")
env = dbutils.widgets.get("env")

In [ ]:
%run ../setup/00_setup_parameters 

In [ ]:
import bronze_schemas
from pyspark.sql import functions as F
import importlib

importlib.reload(bronze_schemas)

def ingest_to_bronze():
    bronze_streame_source_schema = bronze_schemas.get_bronze_schema_for_stream_source()

    # 1. Read stream from Event Hub (Kafka interface)
    # Event Hubs uses $ConnectionString as the username and the connection string as the password
    sasl_config = f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{eventhub_connection_string}";'

    raw_stream_df = (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", eventhub_bootstrap_server)
        .option("subscribe", eventhub_topic)
        .option("kafka.security.protocol", "SASL_SSL")
        .option("kafka.sasl.mechanism", "PLAIN")
        .option("kafka.sasl.jaas.config", sasl_config)
        .option("kafka.request.timeout.ms", "60000")
        .option("kafka.default.api.timeout.ms", "60000")
        .option("startingOffsets", "earliest")
        .load()
    )

    # 2. Parse JSON payload from Kafka 'value' binary column and handle parsing/schema matching
    # Kafka passes the message body as binary in 'value', so we cast it to string and parse JSON against your schema
    parsed_json_df = raw_stream_df.select(
        F.from_json(F.col("value").cast("string"), bronze_streame_source_schema).alias("parsed_json"),
        F.col("timestamp").alias("_kafka_enqueued_timestamp"),
        F.col("partition").alias("_kafka_partition"),
        F.col("offset").alias("_kafka_offset")
    )

    # Flatten the parsed struct columns while preserving audit metadata
    enriched_df = parsed_json_df.select(
        F.col("parsed_json.*"),
        F.col("_kafka_enqueued_timestamp"),
        F.col("_kafka_partition"),
        F.col("_kafka_offset")
    ).withColumn("insert_timestamp", F.current_timestamp())

    # 3. Handle data validation / quarantine filtering
    # Check if a rescue or validation structure exists, otherwise filter on trade_id
    if "trade_id" in enriched_df.columns:
        clean_df = enriched_df.filter(F.col("trade_id").isNotNull())
        quarantine_df = enriched_df.filter(F.col("trade_id").isNull())
    else:
        clean_df = enriched_df
        quarantine_df = enriched_df.limit(0) # Empty dataframe fallback for quarantine if schema strictly aligns

    bronze_checkpoint = checkpoint_path + "/bronze_streaming_source"
    quarantine_checkpoint = checkpoint_path + "/bronze_quarantine"

    # 4. Write Clean Data to Main Bronze Table
    clean_query = (
        clean_df
        .writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", bronze_checkpoint)
        .trigger(availableNow=True)
        .toTable(f"{catalog}.{schema}.bronze_streaming_source")
    )

    # 5. Write Bad Data to Quarantine Table
    quarantine_query = (
        quarantine_df
        .writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", quarantine_checkpoint)
        .trigger(availableNow=True)
        .toTable(f"{catalog}.{schema}.bronze_quarantined_records")
    )

    clean_query.awaitTermination()
    quarantine_query.awaitTermination()

ingest_to_bronze()